# Baseline - Standalone MPNet with Paraphrased Data

This notebook performs evaluation on the MPNet model without further training. The input data for this notebook is embedded data output from the `Text Embedding` notebook. Both the original and the paraphrased version of the data will be used.

# Set up

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

def f1(x,y,thres):
    yx = np.zeros(x.shape[0])
    yy = np.ones(y.shape[0])
    yhx = np.zeros(x.shape[0])
    yhx[x > thres] = 1
    yhy = np.zeros(y.shape[0])
    yhy[y > thres] = 1
    return f1_score(np.concatenate([yx,yy]), np.concatenate([yhx,yhy]))

In [ ]:
def run_model(answers, gpt1, gpt2, answers_p, gpt1_p, gpt2_p):
  valid_accuracies = []
  valid_f1s = []
  test_accuracies = []
  test_f1s = []

  for seed in (1111,2222,3333,4444,5555,6666,7777,8888,9999,1010):
    np.random.seed(seed)

    #train-test-validation split
    sampling_indices = np.random.uniform(0,1,answers.shape[0])
    
    va = answers[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg1 = gpt1[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg2 = gpt2[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vap = answers_p[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg1p = gpt1_p[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]
    vg2p = gpt2_p[(0.8 <= sampling_indices) & (sampling_indices < 0.9)]

    valid_answers = np.vstack([va,va,va,va,va,va,vap,vap,vap,vap,vap,vap])
    valid_gpt1 = np.vstack([vg1,vg1,vg1,vg2,vg2,vg1p,vg1,vg1,vg1,vg2,vg2,vg1p])
    valid_gpt2 = np.vstack([vg2,vg1p,vg2p,vg1p,vg2p,vg2p,vg2,vg1p,vg2p,vg1p,vg2p,vg2p])

    test_answers = answers_p[sampling_indices >= 0.9]
    test_gpt1 = gpt1_p[sampling_indices >= 0.9]
    test_gpt2 = gpt2_p[sampling_indices >= 0.9]

    #valid accuracy
    valid_gpt_distances_org = ((valid_gpt1 - valid_gpt2)**2).sum(axis=1)
    valid_answer_distances_org = ((valid_gpt1 - valid_answers)**2).sum(axis=1)
    valid_accuracies.append((valid_gpt_distances_org < valid_answer_distances_org).mean())

    #tune threshold
    thresholds = []
    perf = []
    for thres in np.arange(0,0.5,0.01):
      thresholds.append(thres)
      perf.append(f1(valid_gpt_distances_org, valid_answer_distances_org, thres))

    emb_thres = thresholds[perf.index(max(perf))]
    emb_thres, max(perf)

    #valid f1
    valid_f1s.append(max(perf))

    #test accuracy
    test_gpt_distances_org = ((test_gpt1 - test_gpt2)**2).sum(axis=1)
    test_answer_distances_org = ((test_gpt1 - test_answers)**2).sum(axis=1)
    test_accuracies.append((test_gpt_distances_org < test_answer_distances_org).mean())

    #test f1
    test_f1s.append(f1(test_gpt_distances_org, test_answer_distances_org, emb_thres))
  return valid_accuracies, test_accuracies, valid_f1s, test_f1s

# Load Data

We will load the embeddings of both the original data and paraphrased data.

In [ ]:
dataset = 'SQUAD_GPT4'

import pickle

with open(f'Data/{dataset}_answer_embs.obj', 'rb') as f:
  answers = pickle.load(f)
with open(f'Data/{dataset}_gpt1_embs.obj', 'rb') as f:
  gpt1 = pickle.load(f)
with open(f'Data/{dataset}_gpt2_embs.obj', 'rb') as f:
  gpt2 = pickle.load(f)

with open(f'Data/{dataset}_paraphrased_answer_embs.obj', 'rb') as f:
  answers_p = pickle.load(f)
with open(f'Data/{dataset}_paraphrased_gpt1_embs.obj', 'rb') as f:
  gpt1_p = pickle.load(f)
with open(f'Data/{dataset}_paraphrased_gpt2_embs.obj', 'rb') as f:
  gpt2_p = pickle.load(f)

# Run Model

In [ ]:
valid_accuracies, test_accuracies, valid_f1s, test_f1s = run_model(answers, gpt1, gpt2, answers_p, gpt1_p, gpt2_p)

In [ ]:
np.mean(valid_accuracies), np.mean(test_accuracies), np.mean(valid_f1s), np.mean(test_f1s)